# NB03 — Ablation Studies for BTCS
**Andre da Costa Silva | ITA | 2026**

Cinco ablacoes para validar decisoes de design do BTCS:
1. **delta_L sweep** {1, 3, 7, 14, 30}: sensibilidade a janela temporal
2. **Budget B sweep** {50, 100, 200, 500, None}: tradeoff OCR vs coverage
3. **Arquitetura**: WCC-only vs Leiden-only vs WCC+Leiden (hierarquico)
4. **Score-weights vs Uniform** (GAP 2): pesos GNN/temporal no Lk vs pesos uniformes
5. **Louvain truncation cost** (GAP 3): coverage/purity perdida no truncamento pos-hoc

Foco em k={1%, 5%} x {AML100k, AML1M} para manter runtime viavel.

Executar: celulas 1→2→3→3b→4→5→6→7→8→9→10→11→12→13 (ou Runtime→Run all)

In [ ]:
# CELULA 1 - Setup (identico ao nb02)
import os, sys, subprocess, time, math, contextlib
from pathlib import Path
from collections import defaultdict

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil', 'tqdm',
     'python-igraph', 'leidenalg')
try:
    import torch
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | igraph {ig.__version__} | leidenalg {leidenalg.version}')
print(f'device: {DEVICE}')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive: {e}')

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()

AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML100K_ARTIF = AML100K_BASE / 'artifacts'
AML100K_PROBS = AML100K_BASE / 'results/probs_v4'
AML100K_MODEL = 'SAGE'
AML100K_SEED  = 42

AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'
AML1M_ARTIF   = AML1M_BASE / 'artifacts'
AML1M_PROBS   = AML1M_BASE / 'results_aml1m_graphsage_only/probs_v4'
AML1M_MODEL   = 'GraphSAGE'
AML1M_SEED    = 44

ABL_OUT = BASE / 'GrafosGNN/results/nb03_ablations'
ABL_OUT.mkdir(parents=True, exist_ok=True)
NB01_OUT = BASE / 'GrafosGNN/results/nb01_baselines'

print('\n=== Verificacao ===')
for label, path in [
    ('AML100k cache', AML100K_ARTIF / 'edge_data_v4_clean.pt'),
    ('AML100k probs', AML100K_PROBS / f'{AML100K_MODEL}_seed{AML100K_SEED}_test.npz'),
    ('AML1M   cache', AML1M_ARTIF   / 'edge_data_v4_clean.pt'),
    ('AML1M   probs', AML1M_PROBS   / f'{AML1M_MODEL}_seed{AML1M_SEED}_test.npz'),
    ('nb01 results', NB01_OUT / 'b0_b1_b2_b3_results.csv'),
]:
    print(f"  {'ok' if path.exists() else 'MISS'}  {label}")

In [ ]:
# CELULA 2 - Funcoes base (identico ao nb02)

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter()
    r  = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb']  = proc.memory_info().rss / 1024**2 - m0

def evaluate_cases(cases, gt, ranked, k):
    y_full = np.asarray(gt['y_full'], dtype=int)
    y_test = np.asarray(ranked['y'],  dtype=int)
    pos_te = int(y_test.sum())
    K      = max(1, int(math.ceil(k * len(y_test))))
    nan_row = {m: np.nan for m in [
        'n_cases','coverage','purity_induced','cr_at_k','recall_in_cases',
        'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
        'e_ind_total','e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}
    if not cases:
        return nan_row
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    non_empty = [c['induced_edges'] for c in cases if c['induced_edges']]
    if non_empty:
        all_ind = np.unique(np.concatenate(
            [np.asarray(x, dtype=np.int64) for x in non_empty]))
    else:
        all_ind = np.array([], dtype=np.int64)
    pos_ind  = int(y_full[all_ind].sum()) if len(all_ind) > 0 else 0
    coverage = float(pos_ind / max(pos_te, 1))
    purity   = float(pos_ind / max(int(ind_sizes.sum()), 1))
    pos_sel  = sum(int(y_test[c['seed_edges']].sum()) for c in cases if c['seed_edges'])
    recall   = float(pos_sel / max(pos_te, 1))
    cr_at_k  = float(recall / max(coverage, 1e-12)) if coverage > 0 else np.nan
    return {'n_cases':len(cases), 'coverage':coverage, 'purity_induced':purity,
            'cr_at_k':cr_at_k, 'recall_in_cases':recall,
            'edges_per_case_median':float(np.median(ind_sizes)),
            'edges_per_case_p95':float(np.percentile(ind_sizes, 95)),
            'edges_per_case_max':float(ind_sizes.max()),
            'e_ind_total':int(ind_sizes.sum()),
            'e_ind_over_ek':float(ind_sizes.sum() / max(K, 1)),
            'ocr_b100':float((ind_sizes>100).mean()),
            'ocr_b500':float((ind_sizes>500).mean()),
            'ocr_b1000':float((ind_sizes>1000).mean())}

def load_dataset_artifacts(artif_dir, probs_dir, model, seed, dataset_name=''):
    artif_dir = Path(artif_dir); probs_dir = Path(probs_dir)
    npz  = np.load(probs_dir / f'{model}_seed{seed}_test.npz')
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] {len(p_te):,} arestas teste, {y_te.sum():,} pos ({100*y_te.mean():.2f}%)')
    cache  = torch.load(artif_dir/'edge_data_v4_clean.pt', map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src_te = ei_all[0, te_idx]; dst_te = ei_all[1, te_idx]
    assert len(p_te)==len(src_te), f'Mismatch: {len(p_te)} vs {len(src_te)}'
    print(f'[{dataset_name}] Cache: {ei_all.shape[1]:,} totais, {len(te_idx):,} teste')
    return ({'scores':p_te,'y':y_te,'src':src_te,'dst':dst_te},
            {'src':src_te,'dst':dst_te},
            {'y_full':y_te}, te_idx, ei_all)

def load_timestamps_from_csv(data_dir, te_idx, ei_all):
    data_dir = Path(data_dir)
    candidates = ['transactions.csv','transaction.csv',
                  'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
                  'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv']
    csv_path = next((list(data_dir.rglob(c))[0] for c in candidates
                     if list(data_dir.rglob(c))), None)
    if csv_path is None:
        raise FileNotFoundError(f'CSV nao encontrado em {data_dir}')
    print(f'  CSV: {csv_path.name}')
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next((c for c in ['timestamp','time','date','datetime','step'] if c in df.columns), None)
    src_col  = next((c for c in ['sender_account_id','src','source','from','sender','src_id'] if c in df.columns), None)
    dst_col  = next((c for c in ['receiver_account_id','dst','dest','to','receiver','dst_id'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Coluna tempo nao encontrada. Cols: {list(df.columns)}')
    print(f'  Colunas: time={time_col!r}  src={src_col!r}  dst={dst_col!r}')
    if pd.api.types.is_numeric_dtype(df[time_col]):
        ts_raw = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    else:
        ts_raw = (pd.to_datetime(df[time_col], errors='coerce')
                    .fillna(pd.Timestamp('1970-01-01')).astype(np.int64)).values
    order = np.argsort(ts_raw, kind='stable')
    ts_sort = ts_raw[order]
    if src_col and dst_col:
        mask = df[src_col].astype(str).values[order] != df[dst_col].astype(str).values[order]
        ts_clean = ts_sort[mask]
        n_loops = int((~mask).sum())
        if n_loops > 0: print(f'  Self-loops removidos: {n_loops}')
    else:
        ts_clean = ts_sort
    n_csv, n_cache = len(ts_clean), ei_all.shape[1]
    if n_csv != n_cache:
        raise AssertionError(f'Mismatch: {n_csv} ts vs {n_cache} arestas')
    ts_test = ts_clean[te_idx]
    print(f'  Timestamps: [{ts_test.min()}, {ts_test.max()}]')
    return ts_test

print('Funcoes base definidas.')

In [ ]:
# CELULA 3 - BTCS multi-modo para ablacoes
#
# mode='wcc_leiden' : v3 hierarquico (WCC + Leiden para grandes)
# mode='wcc_only'   : WCC + budget cap direto (sem Leiden)
# mode='leiden_only' : Leiden global em Lk + budget cap (= v2)

def build_Lk(src_sel, dst_sel, ts_sel, delta_L, max_hub_edges=500, verbose=True):
    """Constroi grafo auxiliar Lk."""
    K = len(src_sel)
    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))
    lk_edges = set()
    n_hubs_capped = 0
    for node, elist in node_map.items():
        n = len(elist)
        if n < 2:
            continue
        if n > max_hub_edges:
            n_hubs_capped += 1
            elist.sort(key=lambda x: x[1])
            elist = elist[:max_hub_edges]
        else:
            elist.sort(key=lambda x: x[1])
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    lk_edges.add((min(ei, ej), max(ei, ej)))
    if verbose and n_hubs_capped > 0:
        print(f'    Lk: {n_hubs_capped} hubs capped at {max_hub_edges}')
    return list(lk_edges)


def _induced_and_budget(cases, sf, df_, scores, max_node, budget_B):
    """Calcula arestas induzidas (vetorizado) e aplica budget cap."""
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            gid_of[nid] = g
    g_src = np.where(sf < max_node, gid_of[sf], -1)
    g_dst = np.where(df_ < max_node, gid_of[df_], -1)
    mask  = (g_src == g_dst) & (g_src >= 0)
    idx_f = np.where(mask)[0]
    if len(idx_f) > 0:
        gf     = g_src[idx_f]
        order  = np.argsort(gf, kind='stable')
        g_sorted = gf[order]
        i_sorted = idx_f[order]
        uq, cnt  = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    n_capped = 0
    if budget_B is not None and budget_B > 0:
        for case in cases:
            if len(case['induced_edges']) > budget_B:
                n_capped += 1
                idx_arr = np.array(case['induced_edges'], dtype=np.int64)
                sc_ind = scores[idx_arr]
                top_b = idx_arr[np.argsort(-sc_ind)[:budget_B]]
                case['induced_edges'] = top_b.tolist()
    return cases, n_capped


def build_btcs_cases(ranked, full, k=0.01, delta_L=7, resolution=1.0,
                     budget_B=100, seed=42, mode='wcc_leiden'):
    """BTCS multi-modo.
    mode='wcc_leiden' : v3 hierarquico
    mode='wcc_only'   : WCC + budget (sem Leiden)
    mode='leiden_only' : Leiden global em Lk (= v2)
    """
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'], dtype=np.int64)
    sf     = np.asarray(full['src'],      dtype=np.int64)
    df_    = np.asarray(full['dst'],      dtype=np.int64)
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_sel, dst_sel, ts_sel = src[sel], dst[sel], ts[sel]
    max_node = int(max(sf.max(), df_.max(), src_sel.max(), dst_sel.max())) + 1
    print(f'  K={K:,} | mode={mode}')

    # ───────────────────────────────────────────────
    # MODE: leiden_only (= v2)
    # ───────────────────────────────────────────────
    if mode == 'leiden_only':
        lk_edges = build_Lk(src_sel, dst_sel, ts_sel, delta_L)
        print(f'  Lk: {len(lk_edges):,} edges')
        if lk_edges:
            g_lk = ig.Graph(n=K, edges=lk_edges, directed=False)
            g_lk.simplify()
            part = leidenalg.find_partition(
                g_lk, leidenalg.RBConfigurationVertexPartition,
                resolution_parameter=resolution, seed=seed)
            membership = np.array(part.membership, dtype=np.int64)
        else:
            membership = np.arange(K, dtype=np.int64)
        n_cases = int(membership.max()) + 1
        cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
                 for _ in range(n_cases)]
        for i in range(K):
            g = int(membership[i])
            cases[g]['seed_edges'].append(int(sel[i]))
            cases[g]['nodes'].update([int(src_sel[i]), int(dst_sel[i])])
        cases = [c for c in cases if c['nodes']]
        cases, n_capped = _induced_and_budget(cases, sf, df_, scores, max_node, budget_B)
        ind_sizes = np.array([len(c['induced_edges']) for c in cases])
        print(f'  Leiden: {len(cases)} cases | capped={n_capped} | '
              f'median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
        return cases

    # ───────────────────────────────────────────────
    # WCC no subgrafo de nos (comum a wcc_only e wcc_leiden)
    # ───────────────────────────────────────────────
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    n_compact = len(all_nodes)
    edges_compact = [(nmap[int(s)], nmap[int(d)])
                     for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=n_compact, edges=edges_compact, directed=False)
    g_node.simplify()
    wcc = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc = int(wcc_mem.max()) + 1
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel], dtype=np.int64)

    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets  = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])

    # Contar induzidas por WCC
    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes:
            wcc_gid[nid] = w
    g_src_w = np.where(sf < max_node, wcc_gid[sf], -1)
    g_dst_w = np.where(df_ < max_node, wcc_gid[df_], -1)
    mask_w  = (g_src_w == g_dst_w) & (g_src_w >= 0)
    idx_w   = np.where(mask_w)[0]
    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if len(idx_w) > 0:
        gw = g_src_w[idx_w]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt

    # ───────────────────────────────────────────────
    # MODE: wcc_only (WCC + budget, sem Leiden)
    # ───────────────────────────────────────────────
    if mode == 'wcc_only':
        cases = []
        for w in range(n_wcc):
            if not wcc_edge_lists[w]:
                continue
            cases.append({
                'nodes': wcc_node_sets[w],
                'seed_edges': [int(sel[i]) for i in wcc_edge_lists[w]],
                'induced_edges': []
            })
        cases, n_capped = _induced_and_budget(cases, sf, df_, scores, max_node, budget_B)
        ind_sizes = np.array([len(c['induced_edges']) for c in cases])
        print(f'  WCC-only: {len(cases)} cases | capped={n_capped} | '
              f'median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
        return cases

    # ───────────────────────────────────────────────
    # MODE: wcc_leiden (v3 hierarquico)
    # ───────────────────────────────────────────────
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0
    n_kept = 0
    n_split = 0

    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp:
            continue
        need_split = (budget_B is not None
                      and wcc_ind_count[w] > budget_B
                      and len(comp) >= 2)
        if not need_split:
            for i in comp:
                final_mem[i] = next_id
            next_id += 1
            n_kept += 1
        else:
            n_split += 1
            comp_arr = np.array(comp, dtype=np.int64)
            lk_local = build_Lk(
                src_sel[comp_arr], dst_sel[comp_arr], ts_sel[comp_arr],
                delta_L, verbose=False)
            if not lk_local:
                for i in comp:
                    final_mem[i] = next_id
                    next_id += 1
            else:
                n_local = len(comp)
                g_local = ig.Graph(n=n_local, edges=lk_local, directed=False)
                g_local.simplify()
                part = leidenalg.find_partition(
                    g_local, leidenalg.RBConfigurationVertexPartition,
                    resolution_parameter=resolution, seed=seed)
                local_mem = np.array(part.membership, dtype=np.int64)
                n_sub = int(local_mem.max()) + 1
                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub

    n_total = next_id
    # Voto majoritario para nos disjuntos
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0:
            continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_total)]
    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)
    for i in range(K):
        g = int(final_mem[i])
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]

    cases, n_capped = _induced_and_budget(cases, sf, df_, scores, max_node, budget_B)
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  WCC+Leiden: {n_kept} kept + {n_split} split → {len(cases)} cases | '
          f'capped={n_capped} | median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
    return cases

print('BTCS multi-modo (wcc_leiden / wcc_only / leiden_only) definido.')

In [ ]:
# CELULA 3b - build_Lk_weighted + build_btcs_cases com use_scores (GAP 2)
# Redefine build_btcs_cases adicionando suporte a pesos score-informed no Lk.
# use_scores=False (default) → comportamento original identico.
# use_scores=True  → w_ij = exp(-dt/tau) × (1+alpha*min(si,sj)) × 1/(1+beta*log(1+deg))

def build_Lk_weighted(src_sel, dst_sel, ts_sel, scores_sel, delta_L,
                      tau=None, alpha=1.0, beta=1.0,
                      max_hub_edges=500, verbose=True):
    """Constroi Lk com pesos: temporal decay × GNN score boost × hub penalty.

    w_ij = exp(-|ti-tj|/tau) * (1 + alpha*min(si,sj)) * 1/(1 + beta*log(1+deg))
    Retorna (edges_list, weights_list) — sem duplicatas (mantém max peso).
    """
    K = len(src_sel)
    if tau is None:
        tau = float(max(delta_L, 1))

    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append((i, float(ts_sel[i]), float(scores_sel[i])))
        node_map[int(dst_sel[i])].append((i, float(ts_sel[i]), float(scores_sel[i])))

    lk_edge_weights = {}   # (min_i, max_j) -> max_weight
    n_hubs_capped = 0

    for node, elist in node_map.items():
        n = len(elist)
        if n < 2:
            continue
        if n > max_hub_edges:
            n_hubs_capped += 1
            elist.sort(key=lambda x: x[1])
            elist = elist[:max_hub_edges]
        else:
            elist.sort(key=lambda x: x[1])

        deg = len(elist)
        hub_pen = 1.0 / (1.0 + beta * math.log1p(float(deg)))

        for i in range(len(elist)):
            ei, ti, si = elist[i]
            for j in range(i + 1, len(elist)):
                ej, tj, sj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    key = (min(int(ei), int(ej)), max(int(ei), int(ej)))
                    dt = tj - ti
                    temp_decay  = math.exp(-dt / tau)
                    score_boost = 1.0 + alpha * min(si, sj)
                    w = temp_decay * score_boost * hub_pen
                    if key not in lk_edge_weights or lk_edge_weights[key] < w:
                        lk_edge_weights[key] = w

    if verbose and n_hubs_capped > 0:
        print(f'    Lk (weighted): {n_hubs_capped} hubs capped at {max_hub_edges}')

    edges   = list(lk_edge_weights.keys())
    weights = [lk_edge_weights[e] for e in edges]
    return edges, weights


def build_btcs_cases(ranked, full, k=0.01, delta_L=7, resolution=1.0,
                     budget_B=100, seed=42, mode='wcc_leiden', use_scores=False):
    """BTCS multi-modo + suporte a pesos score-informed.

    mode : 'wcc_leiden' (v3, default) | 'wcc_only' | 'leiden_only'
    use_scores : False → pesos uniformes (original)
                 True  → pesos w_ij via build_Lk_weighted
    """
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'], dtype=np.int64)
    sf     = np.asarray(full['src'],      dtype=np.int64)
    df_    = np.asarray(full['dst'],      dtype=np.int64)
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_sel    = src[sel]
    dst_sel    = dst[sel]
    ts_sel     = ts[sel]
    scores_sel = scores[sel]
    max_node   = int(max(sf.max(), df_.max(), src_sel.max(), dst_sel.max())) + 1
    print(f'  K={K:,} | mode={mode} | use_scores={use_scores}')

    def _build_lk(s, d, t, sc):
        if use_scores:
            return build_Lk_weighted(s, d, t, sc, delta_L, verbose=False)
        else:
            edges = build_Lk(s, d, t, delta_L, verbose=False)
            return edges, None  # None → uniform

    def _partition(n_nodes, edges, weights):
        g = ig.Graph(n=n_nodes, edges=edges, directed=False)
        if weights is None:
            g.simplify()
            part = leidenalg.find_partition(
                g, leidenalg.RBConfigurationVertexPartition,
                resolution_parameter=resolution, seed=seed)
        else:
            # Edges already deduplicated by build_Lk_weighted — no simplify needed
            g.es['weight'] = weights
            part = leidenalg.find_partition(
                g, leidenalg.RBConfigurationVertexPartition,
                weights='weight',
                resolution_parameter=resolution, seed=seed)
        return np.array(part.membership, dtype=np.int64)

    # ─── leiden_only ───
    if mode == 'leiden_only':
        lk_edges, lk_w = _build_lk(src_sel, dst_sel, ts_sel, scores_sel)
        print(f'  Lk: {len(lk_edges):,} edges')
        membership = _partition(K, lk_edges, lk_w) if lk_edges else np.arange(K, dtype=np.int64)
        n_cases = int(membership.max()) + 1
        cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_cases)]
        for i in range(K):
            g = int(membership[i])
            cases[g]['seed_edges'].append(int(sel[i]))
            cases[g]['nodes'].update([int(src_sel[i]), int(dst_sel[i])])
        cases = [c for c in cases if c['nodes']]
        cases, n_capped = _induced_and_budget(cases, sf, df_, scores, max_node, budget_B)
        ind_sizes = np.array([len(c['induced_edges']) for c in cases])
        print(f'  Leiden: {len(cases)} cases | capped={n_capped} | '
              f'median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
        return cases

    # ─── WCC base (comum a wcc_only e wcc_leiden) ───
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    n_compact = len(all_nodes)
    edges_compact = [(nmap[int(s)], nmap[int(d)]) for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=n_compact, edges=edges_compact, directed=False)
    g_node.simplify()
    wcc     = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc   = int(wcc_mem.max()) + 1
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel], dtype=np.int64)

    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets  = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])

    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes: wcc_gid[nid] = w
    g_src_w = np.where(sf < max_node, wcc_gid[sf], -1)
    g_dst_w = np.where(df_ < max_node, wcc_gid[df_], -1)
    mask_w  = (g_src_w == g_dst_w) & (g_src_w >= 0)
    idx_w   = np.where(mask_w)[0]
    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if len(idx_w) > 0:
        gw = g_src_w[idx_w]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt

    # ─── wcc_only ───
    if mode == 'wcc_only':
        cases = []
        for w in range(n_wcc):
            if not wcc_edge_lists[w]: continue
            cases.append({'nodes': wcc_node_sets[w],
                          'seed_edges': [int(sel[i]) for i in wcc_edge_lists[w]],
                          'induced_edges': []})
        cases, n_capped = _induced_and_budget(cases, sf, df_, scores, max_node, budget_B)
        ind_sizes = np.array([len(c['induced_edges']) for c in cases])
        print(f'  WCC-only: {len(cases)} cases | capped={n_capped} | '
              f'median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
        return cases

    # ─── wcc_leiden (v3 hierarquico) ───
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0; n_kept = 0; n_split = 0

    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp: continue
        need_split = (budget_B is not None and wcc_ind_count[w] > budget_B and len(comp) >= 2)
        if not need_split:
            for i in comp: final_mem[i] = next_id
            next_id += 1; n_kept += 1
        else:
            n_split += 1
            comp_arr = np.array(comp, dtype=np.int64)
            lk_local, lk_w = _build_lk(
                src_sel[comp_arr], dst_sel[comp_arr],
                ts_sel[comp_arr],  scores_sel[comp_arr])
            if not lk_local:
                for i in comp: final_mem[i] = next_id; next_id += 1
            else:
                local_mem = _partition(len(comp), lk_local, lk_w)
                n_sub = int(local_mem.max()) + 1
                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub

    n_total = next_id
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0: continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_total)]
    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)
    for i in range(K):
        g = int(final_mem[i])
        if g >= 0: cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]
    cases, n_capped = _induced_and_budget(cases, sf, df_, scores, max_node, budget_B)
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  WCC+Leiden: {n_kept} kept + {n_split} split → {len(cases)} cases | '
          f'capped={n_capped} | median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
    return cases

print('build_Lk_weighted + build_btcs_cases (use_scores support) definidos.')
print('Celulas 5-7 funcionam sem mudanca (use_scores=False default).')

In [ ]:
# CELULA 4 - Carregar dados + timestamps + baselines
print('Carregando AML100k...')
ranked_100k, full_100k, gt_100k, te_idx_100k, ei_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS, AML100K_MODEL, AML100K_SEED, 'AML100k')
ts_100k = load_timestamps_from_csv(AML100K_BASE, te_idx_100k, ei_100k)
ranked_100k['timestamps'] = ts_100k
full_100k['timestamps']   = ts_100k

print('\nCarregando AML1M...')
ranked_1m, full_1m, gt_1m, te_idx_1m, ei_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS, AML1M_MODEL, AML1M_SEED, 'AML1M')
ts_1m = load_timestamps_from_csv(AML1M_BASE, te_idx_1m, ei_1m)
ranked_1m['timestamps'] = ts_1m
full_1m['timestamps']   = ts_1m

df_baselines = pd.read_csv(NB01_OUT / 'b0_b1_b2_b3_results.csv')
print(f'\nnb01 baselines: {len(df_baselines)} linhas')

DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]
ABL_KS = [0.01, 0.05]  # k=1% e k=5% para ablacoes
print(f'Ablation k values: {ABL_KS}')
print('Dados prontos!')

In [ ]:
# CELULA 5 - Ablacao 1: delta_L sweep
# Fixa: gamma=1.0, B=100, mode=wcc_leiden
# Varia: delta_L in {1, 3, 7, 14, 30}
DELTA_LS = [1, 3, 7, 14, 30]
GAMMA_FIX = 1.0
B_FIX = 100

rows_dl = []
for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*40}  {ds} — delta_L sweep  {"="*40}')
    for dl in DELTA_LS:
        for k in ABL_KS:
            kp = f'{int(k*100)}%'
            print(f'  delta_L={dl} k={kp}:')
            with measure_resources() as res:
                cases = build_btcs_cases(
                    ranked, full, k=k, delta_L=dl,
                    resolution=GAMMA_FIX, budget_B=B_FIX, mode='wcc_leiden')
                m = evaluate_cases(cases, gt, ranked, k)
            row = {'dataset':ds, 'ablation':'delta_L', 'param':dl,
                   'k%':kp, 'k_frac':k, 'resolution':GAMMA_FIX,
                   'budget_B':B_FIX, **m, **res}
            rows_dl.append(row)
            print(f"    dL={dl}: cov={m['coverage']:.3f} pur={m['purity_induced']:.4f} "
                  f"OCR100={m['ocr_b100']:.3f} #cases={m['n_cases']:,} {res['time_s']:.1f}s")

df_abl_dl = pd.DataFrame(rows_dl)
print(f'\nAblacao delta_L: {len(rows_dl)} runs concluidos.')

In [ ]:
# CELULA 6 - Ablacao 2: Budget B sweep
# Fixa: gamma=1.0, delta_L=7, mode=wcc_leiden
# Varia: B in {50, 100, 200, 500, None}
BUDGETS = [50, 100, 200, 500, None]
DELTA_FIX = 7

rows_b = []
for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*40}  {ds} — Budget B sweep  {"="*40}')
    for B in BUDGETS:
        for k in ABL_KS:
            kp = f'{int(k*100)}%'
            B_label = f'B={B}' if B else 'NoBudget'
            print(f'  {B_label} k={kp}:')
            with measure_resources() as res:
                cases = build_btcs_cases(
                    ranked, full, k=k, delta_L=DELTA_FIX,
                    resolution=GAMMA_FIX, budget_B=B, mode='wcc_leiden')
                m = evaluate_cases(cases, gt, ranked, k)
            row = {'dataset':ds, 'ablation':'budget_B', 'param':B if B else 0,
                   'k%':kp, 'k_frac':k, 'resolution':GAMMA_FIX,
                   'budget_B':B if B else 0, **m, **res}
            rows_b.append(row)
            print(f"    {B_label}: cov={m['coverage']:.3f} OCR100={m['ocr_b100']:.3f} "
                  f"median_ind={m['edges_per_case_median']:.0f} "
                  f"max_ind={m['edges_per_case_max']:.0f} {res['time_s']:.1f}s")

df_abl_b = pd.DataFrame(rows_b)
print(f'\nAblacao Budget B: {len(rows_b)} runs concluidos.')

In [ ]:
# CELULA 7 - Ablacao 3: Comparacao de arquitetura
# Fixa: gamma=1.0, delta_L=7, B=100
# Varia: mode in {wcc_only, leiden_only, wcc_leiden}
MODES = ['wcc_only', 'leiden_only', 'wcc_leiden']

rows_arch = []
for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*40}  {ds} — Architecture comparison  {"="*40}')
    for mode in MODES:
        for k in ABL_KS:
            kp = f'{int(k*100)}%'
            print(f'  {mode} k={kp}:')
            with measure_resources() as res:
                cases = build_btcs_cases(
                    ranked, full, k=k, delta_L=DELTA_FIX,
                    resolution=GAMMA_FIX, budget_B=B_FIX,
                    mode=mode)
                m = evaluate_cases(cases, gt, ranked, k)
            row = {'dataset':ds, 'ablation':'architecture', 'param':mode,
                   'k%':kp, 'k_frac':k, **m, **res}
            rows_arch.append(row)
            print(f"    {mode}: cov={m['coverage']:.3f} pur={m['purity_induced']:.4f} "
                  f"OCR100={m['ocr_b100']:.3f} #cases={m['n_cases']:,} {res['time_s']:.1f}s")

df_abl_arch = pd.DataFrame(rows_arch)

# Tabela pivot
print('\n=== Comparacao de Arquitetura ===')
for ds in ['AML100k', 'AML1M']:
    print(f'\n--- {ds} ---')
    sub = df_abl_arch[df_abl_arch['dataset']==ds]
    pivot = sub.pivot_table(
        index='k%', columns='param',
        values=['coverage','purity_induced','ocr_b100','n_cases','time_s'],
        aggfunc='first').round(4)
    display(pivot)

print(f'\nAblacao Arquitetura: {len(rows_arch)} runs concluidos.')

In [ ]:
# CELULA 8 - Plots de ablacao
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ─── Plot 1 (top-left): delta_L vs Coverage ───
for col, ds in enumerate(['AML100k', 'AML1M']):
    ax = axes[0, col]
    sub = df_abl_dl[df_abl_dl['dataset']==ds]
    for kp, color, marker in [('1%','#1f77b4','o'), ('5%','#d62728','s')]:
        d = sub[sub['k%']==kp].sort_values('param')
        ax.plot(d['param'], d['coverage'], color=color, marker=marker,
                ms=8, linewidth=2, label=f'k={kp}')
    ax.set_xlabel('delta_L')
    ax.set_ylabel('Coverage')
    ax.set_title(f'{ds} — Coverage vs delta_L')
    ax.set_xticks(DELTA_LS)
    ax.legend()
    ax.grid(alpha=0.3)

# ─── Plot 2 (top-right): Budget B vs Coverage ───
ax = axes[0, 2]
for ds, ls in [('AML100k', '-'), ('AML1M', '--')]:
    sub = df_abl_b[(df_abl_b['dataset']==ds) & (df_abl_b['k%']=='1%')]
    sub = sub.sort_values('param')
    x_vals = sub['param'].replace(0, 999).values  # NoBudget → 999
    ax.plot(x_vals, sub['coverage'], marker='o', ls=ls, linewidth=2,
            label=f'{ds} k=1%')
ax.set_xlabel('Budget B')
ax.set_ylabel('Coverage')
ax.set_title('Coverage vs Budget B (k=1%)')
ax.set_xscale('log')
ax.set_xticks([50, 100, 200, 500, 999])
ax.set_xticklabels(['50', '100', '200', '500', 'None'])
ax.legend()
ax.grid(alpha=0.3)

# ─── Plot 3 (bottom): Architecture comparison bars ───
for col, ds in enumerate(['AML100k', 'AML1M']):
    ax = axes[1, col]
    sub = df_abl_arch[df_abl_arch['dataset']==ds]
    x = np.arange(len(ABL_KS))
    w = 0.25
    mode_labels = {'wcc_only': 'WCC-only', 'leiden_only': 'Leiden-only',
                   'wcc_leiden': 'WCC+Leiden (v3)'}
    mode_colors = {'wcc_only': '#1f77b4', 'leiden_only': '#ff7f0e',
                   'wcc_leiden': '#2ca02c'}
    for j, mode in enumerate(MODES):
        vals = []
        for kv in ABL_KS:
            kp = f'{int(kv*100)}%'
            row = sub[(sub['param']==mode) & (sub['k%']==kp)]
            vals.append(float(row['coverage']) if not row.empty else 0)
        ax.bar(x + j*w, vals, w, label=mode_labels[mode],
               color=mode_colors[mode], alpha=0.85)
    ax.set_title(f'{ds} — Coverage by Architecture')
    ax.set_xticks(x + w)
    ax.set_xticklabels([f'{int(k*100)}%' for k in ABL_KS])
    ax.set_ylabel('Coverage')
    ax.legend(fontsize=8)

# ─── Plot 4 (bottom-right): OCR by architecture ───
ax = axes[1, 2]
for col, ds in enumerate(['AML100k', 'AML1M']):
    sub = df_abl_arch[(df_abl_arch['dataset']==ds) & (df_abl_arch['k%']=='1%')]
    x_offset = col * (len(MODES) + 1)
    for j, mode in enumerate(MODES):
        row = sub[sub['param']==mode]
        if not row.empty:
            ocr = float(row['ocr_b100'])
            color = mode_colors[mode]
            lbl = mode_labels[mode] if col == 0 else ''
            ax.bar(x_offset + j, ocr, color=color, alpha=0.85, label=lbl)
ax.set_title('OCR(100) by Architecture (k=1%)')
ax.set_xticks([1, 1 + len(MODES) + 1])
ax.set_xticklabels(['AML100k', 'AML1M'])
ax.set_ylabel('OCR(B=100)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ABL_OUT / 'ablation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots salvos.')

In [ ]:
# CELULA 13 - Plot e Export: Louvain Truncation Analysis (GAP 3)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

groups_t    = ['AML100k\nk=1%', 'AML100k\nk=5%', 'AML1M\nk=1%', 'AML1M\nk=5%']
group_keys_t = [('AML100k','1%'), ('AML100k','5%'), ('AML1M','1%'), ('AML1M','5%')]

# ─── Plot A: Coverage before vs after truncation ───
ax = axes[0]
w = 0.35; x4 = np.arange(4)
cov_before = []; cov_after = []
for ds, kp in group_keys_t:
    row = df_abl_trunc[(df_abl_trunc['dataset']==ds) & (df_abl_trunc['k%']==kp)]
    cov_before.append(float(row['coverage_no_trunc'])   if not row.empty else 0.0)
    cov_after.append( float(row['coverage_with_trunc'])  if not row.empty else 0.0)

ax.bar(x4 - w/2, cov_before, w, label='Before truncation', color='#1f77b4', alpha=0.85)
ax.bar(x4 + w/2, cov_after,  w, label='After truncation',  color='#d62728', alpha=0.85)
for i in range(4):
    loss = cov_before[i] - cov_after[i]
    if loss > 0:
        ax.text(x4[i], max(cov_before[i], cov_after[i]) + 0.005,
                f'−{loss:.3f}', ha='center', va='bottom', fontsize=8,
                color='#d62728', fontweight='bold')
ax.set_title('Coverage: Louvain Before vs After Budget Truncation', fontsize=10)
ax.set_xticks(x4); ax.set_xticklabels(groups_t, fontsize=9)
ax.set_ylabel('Coverage'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ─── Plot B: Fraction of communities over budget ───
ax = axes[1]
frac_over = []
for ds, kp in group_keys_t:
    row = df_abl_trunc[(df_abl_trunc['dataset']==ds) & (df_abl_trunc['k%']==kp)]
    frac_over.append(float(row['frac_over_budget']) if not row.empty else 0.0)

bars = ax.bar(x4, frac_over, color=['#1f77b4','#aec7e8','#d62728','#ffbb78'], alpha=0.85)
for bar, frac in zip(bars, frac_over):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{100*frac:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Fraction of Louvain Communities Exceeding Budget B=100', fontsize=10)
ax.set_xticks(x4); ax.set_xticklabels(groups_t, fontsize=9)
ax.set_ylabel('Fraction over budget'); ax.grid(axis='y', alpha=0.3)

# ─── Plot C: Purity of over-budget communities before vs after truncation ───
ax = axes[2]
pur_b, pur_a = [], []
for ds, kp in group_keys_t:
    row = df_abl_trunc[(df_abl_trunc['dataset']==ds) & (df_abl_trunc['k%']==kp)]
    pur_b.append(float(row['purity_oversize_before']) if not row.empty else 0.0)
    pur_a.append(float(row['purity_oversize_after'])  if not row.empty else 0.0)

ax.bar(x4 - w/2, pur_b, w, label='Before truncation', color='#ff7f0e', alpha=0.85)
ax.bar(x4 + w/2, pur_a, w, label='After truncation',  color='#2ca02c', alpha=0.85)
ax.set_title('Purity of Over-Budget Louvain Communities', fontsize=10)
ax.set_xticks(x4); ax.set_xticklabels(groups_t, fontsize=9)
ax.set_ylabel('Purity (Induced)'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle('GAP 3 — Louvain Post-Hoc Truncation Cost\n'
             '(delta_L=7, B=100): communities over-budget drop fraud-relevant edges',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(ABL_OUT / 'ablation_louvain_truncation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot salvo: {ABL_OUT}/ablation_louvain_truncation.png')

# ─── Export ───
df_abl_trunc.to_csv(ABL_OUT / 'ablation_louvain_truncation.csv', index=False)
trunc_cols = ['dataset','k%','n_communities','n_over_budget','frac_over_budget',
              'coverage_no_trunc','coverage_with_trunc','coverage_loss_abs',
              'coverage_loss_pct','pos_dropped',
              'purity_oversize_before','purity_oversize_after']
df_abl_trunc[trunc_cols].round(4).to_latex(
    ABL_OUT / 'ablation_louvain_truncation.tex', index=False, escape=False)

# ─── Append to master ───
df_master = pd.read_csv(ABL_OUT / 'ablation_results.csv')
df_master_updated = pd.concat([df_master, df_abl_trunc], ignore_index=True)
df_master_updated.to_csv(ABL_OUT / 'ablation_results.csv', index=False)

print(f'CSV: {ABL_OUT}/ablation_louvain_truncation.csv ({len(df_abl_trunc)} rows)')
print(f'TEX: {ABL_OUT}/ablation_louvain_truncation.tex')
print(f'Master CSV: ablation_results.csv ({len(df_master_updated)} rows total)')
print('\nProgresso:')
print('1.[ok] nb00  2.[ok] nb01  3.[v3] nb02-BTCS  4.[ok] nb03 (5 ablacoes)')

In [ ]:
# CELULA 12 - Ablacao 5: Custo de truncamento do Louvain (GAP 3)
#
# Hipotese: Louvain cria comunidades sem restricao de orcamento.
# Quando comunidades excedem B, o truncamento (manter top-B por score)
# descarta arestas positivas (fraude), reduzindo cobertura efetiva.
# BTCS incorpora o budget DURANTE a deteccao → OCR(B=100)=0 by design.
#
# Analise: para cada comunidade Louvain que excede B=100:
#   - Pureza e cobertura ANTES do truncamento
#   - Pureza e cobertura APOS o truncamento (top-B por score)
#   - Positivas descartadas (fraudes perdidas)

def analyze_louvain_truncation(ranked, full, gt, k, delta_L=7, budget_B=100, seed=42):
    """Mede custo de truncamento do Louvain para um dado k e dataset."""
    scores = np.asarray(ranked['scores'],     dtype=float)
    src    = np.asarray(ranked['src'],         dtype=np.int64)
    dst    = np.asarray(ranked['dst'],         dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'],  dtype=np.int64)
    sf     = np.asarray(full['src'],           dtype=np.int64)
    df_    = np.asarray(full['dst'],           dtype=np.int64)
    y_full = np.asarray(ranked['y'],           dtype=int)    # proxy: y_test
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_sel, dst_sel, ts_sel = src[sel], dst[sel], ts[sel]
    pos_total_test = int(y_full.sum())

    # Build Lk + run Louvain (community_multilevel, uniform weights — same as B2)
    lk_edges = build_Lk(src_sel, dst_sel, ts_sel, delta_L)
    g_lk = ig.Graph(n=K, edges=lk_edges, directed=False)
    g_lk.simplify()
    clusters   = g_lk.community_multilevel(weights=None)
    membership = np.array(clusters.membership, dtype=np.int64)
    n_communities = int(membership.max()) + 1

    max_node = int(max(sf.max(), df_.max(), src_sel.max(), dst_sel.max())) + 1

    # Build community node sets + seed lists
    comm_nodes = [set() for _ in range(n_communities)]
    for i in range(K):
        g = int(membership[i])
        comm_nodes[g].update([int(src_sel[i]), int(dst_sel[i])])

    # Compute induced edges per community (vectorised)
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, nodes in enumerate(comm_nodes):
        for nid in nodes: gid_of[nid] = g
    g_src = np.where(sf < max_node, gid_of[sf], -1)
    g_dst = np.where(df_ < max_node, gid_of[df_], -1)
    mask  = (g_src == g_dst) & (g_src >= 0)
    idx_f = np.where(mask)[0]

    comm_induced = [[] for _ in range(n_communities)]
    if len(idx_f) > 0:
        gf = g_src[idx_f]
        order = np.argsort(gf, kind='stable')
        g_sorted = gf[order]; i_sorted = idx_f[order]
        uq, cnt = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            comm_induced[g_id] = grp.tolist()

    # Analyse per-community truncation cost
    n_over = 0
    pos_before_set = set()
    pos_after_set  = set()
    purity_before_list = []
    purity_after_list  = []
    size_before_list   = []
    size_over_list     = []

    for g in range(n_communities):
        ind = np.array(comm_induced[g], dtype=np.int64)
        if len(ind) == 0: continue

        pos_ind = np.where(y_full[ind] == 1)[0]
        pos_before_set.update(ind[pos_ind].tolist())
        size_before_list.append(len(ind))
        purity_b = len(pos_ind) / len(ind)

        if len(ind) > budget_B:
            n_over += 1
            size_over_list.append(len(ind))
            # Truncate: keep top-B by GNN score
            sc_ind = scores[ind]
            keep   = ind[np.argsort(-sc_ind)[:budget_B]]
            pos_keep = np.where(y_full[keep] == 1)[0]
            pos_after_set.update(keep[pos_keep].tolist())
            pur_after = len(pos_keep) / budget_B
            purity_before_list.append(purity_b)
            purity_after_list.append(pur_after)
        else:
            pos_after_set.update(ind[pos_ind].tolist())

    cov_before    = len(pos_before_set) / max(pos_total_test, 1)
    cov_after     = len(pos_after_set)  / max(pos_total_test, 1)
    pos_dropped   = len(pos_before_set - pos_after_set)
    cov_loss_abs  = cov_before - cov_after
    cov_loss_pct  = 100.0 * cov_loss_abs / max(cov_before, 1e-12)

    return {
        'n_communities':         n_communities,
        'n_over_budget':         n_over,
        'frac_over_budget':      n_over / max(n_communities, 1),
        'size_max_over':         max(size_over_list, default=0),
        'size_median_over':      float(np.median(size_over_list)) if size_over_list else 0.0,
        'pos_dropped':           pos_dropped,
        'coverage_no_trunc':     cov_before,
        'coverage_with_trunc':   cov_after,
        'coverage_loss_abs':     cov_loss_abs,
        'coverage_loss_pct':     cov_loss_pct,
        'purity_oversize_before': float(np.mean(purity_before_list)) if purity_before_list else 0.0,
        'purity_oversize_after':  float(np.mean(purity_after_list))  if purity_after_list  else 0.0,
    }

# ─── Run analysis ───
rows_trunc = []
for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*50}  {ds} — Louvain Truncation Analysis  {"="*50}')
    for k in ABL_KS:
        kp = f'{int(k*100)}%'
        print(f'  k={kp}:')
        with measure_resources() as res:
            m = analyze_louvain_truncation(ranked, full, gt, k,
                                           delta_L=DELTA_FIX, budget_B=B_FIX)
        row = {'dataset': ds, 'ablation': 'louvain_truncation',
               'k%': kp, 'k_frac': k, **m, **res}
        rows_trunc.append(row)
        print(f"    communities: {m['n_communities']:,}  over-budget: {m['n_over_budget']:,} "
              f"({100*m['frac_over_budget']:.1f}%)")
        print(f"    coverage: {m['coverage_no_trunc']:.3f} → {m['coverage_with_trunc']:.3f}  "
              f"loss={m['coverage_loss_abs']:.3f} ({m['coverage_loss_pct']:.1f}%)")
        print(f"    positive edges dropped: {m['pos_dropped']:,}")
        print(f"    purity oversize: {m['purity_oversize_before']:.4f} → {m['purity_oversize_after']:.4f}")

df_abl_trunc = pd.DataFrame(rows_trunc)

# ─── Summary table ───
print('\n=== Louvain Truncation Cost Summary ===')
cols_show = ['dataset','k%','n_communities','n_over_budget','frac_over_budget',
             'coverage_no_trunc','coverage_with_trunc','coverage_loss_abs',
             'coverage_loss_pct','pos_dropped']
display(df_abl_trunc[cols_show].round(4))
print(f'\nAblacao Truncamento: {len(rows_trunc)} runs concluidos.')

In [ ]:
# CELULA 11 - Plot e Export: Score-Weight Ablation
import matplotlib.pyplot as plt
import numpy as np

groups    = ['AML100k\nk=1%', 'AML100k\nk=5%', 'AML1M\nk=1%', 'AML1M\nk=5%']
group_keys = [('AML100k','1%'), ('AML100k','5%'), ('AML1M','1%'), ('AML1M','5%')]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
w = 0.35; x4 = np.arange(4)

for col, (metric, mlabel) in enumerate([('coverage', 'Coverage'),
                                          ('purity_induced', 'Purity (Induced)')]):
    ax = axes[col]
    vals_u, vals_w = [], []
    for ds, kp in group_keys:
        su = df_abl_sw[(df_abl_sw['dataset']==ds)&(df_abl_sw['param']=='uniform')        &(df_abl_sw['k%']==kp)]
        sw = df_abl_sw[(df_abl_sw['dataset']==ds)&(df_abl_sw['param']=='score_weighted')&(df_abl_sw['k%']==kp)]
        vals_u.append(float(su[metric]) if not su.empty else 0.0)
        vals_w.append(float(sw[metric]) if not sw.empty else 0.0)

    bars_u = ax.bar(x4 - w/2, vals_u, w, label='Uniform weights',       color='#1f77b4', alpha=0.85)
    bars_w = ax.bar(x4 + w/2, vals_w, w, label='Score-informed weights', color='#d62728', alpha=0.85)

    # Annotate delta
    for i in range(4):
        delta = vals_w[i] - vals_u[i]
        y_top = max(vals_u[i], vals_w[i]) + 0.005
        sign = '+' if delta >= 0 else ''
        ax.text(x4[i], y_top, f'{sign}{delta:.3f}', ha='center', va='bottom',
                fontsize=8, color='#2ca02c' if delta >= 0 else '#ff7f0e', fontweight='bold')

    ax.set_title(f'{mlabel}: Score-informed vs Uniform Lk weights', fontsize=11)
    ax.set_xticks(x4); ax.set_xticklabels(groups, fontsize=9)
    ax.set_ylabel(mlabel); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle('GAP 2 — Ablation: Score-Informed vs Uniform Edge Weights in Lk\n'
             '(wcc_leiden, delta_L=7, B=100)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(ABL_OUT / 'ablation_score_weight.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot salvo: {ABL_OUT}/ablation_score_weight.png')

# ─── Export CSV + LaTeX ───
df_abl_sw.to_csv(ABL_OUT / 'ablation_score_weight.csv', index=False)

sw_cols = ['dataset','param','k%','n_cases','coverage','purity_induced','ocr_b100','time_s']
df_abl_sw[sw_cols].round(4).to_latex(
    ABL_OUT / 'ablation_score_weight.tex', index=False, escape=False)

# ─── Append to master ablation_results.csv ───
df_all_updated = pd.concat(
    [pd.read_csv(ABL_OUT/'ablation_results.csv'), df_abl_sw],
    ignore_index=True)
df_all_updated.to_csv(ABL_OUT / 'ablation_results.csv', index=False)

print(f'CSV: {ABL_OUT}/ablation_score_weight.csv ({len(df_abl_sw)} rows)')
print(f'TEX: {ABL_OUT}/ablation_score_weight.tex')
print(f'Master CSV atualizado: ablation_results.csv ({len(df_all_updated)} rows total)')
print('\nProgresso:')
print('1.[ok] nb00  2.[ok] nb01  3.[v3] nb02-BTCS  4.[ok] nb03-ablacoes (4 ablacoes)')

In [ ]:
# CELULA 10 - Ablacao 4: Score-weights vs Uniform weights no Lk (GAP 2)
#
# Hipotese: pesos w_ij = temp_decay × score_boost × hub_pen melhoram cobertura
# vs Leiden generico (pesos uniformes), porque priorizam arestas de alto score
# que compartilham conta no mesmo janela temporal.
#
# Fixa: mode=wcc_leiden, delta_L=7, B=100, gamma=1.0
# Varia: use_scores in {False (uniform), True (score-informed)}

rows_sw = []

for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*50}  {ds} — Score-Weight Ablation  {"="*50}')
    for use_scores in [False, True]:
        label = 'score_weighted' if use_scores else 'uniform'
        for k in ABL_KS:
            kp = f'{int(k*100)}%'
            print(f'  [{label}] k={kp}:')
            with measure_resources() as res:
                cases = build_btcs_cases(
                    ranked, full, k=k, delta_L=DELTA_FIX,
                    resolution=GAMMA_FIX, budget_B=B_FIX,
                    mode='wcc_leiden', use_scores=use_scores)
                m = evaluate_cases(cases, gt, ranked, k)
            row = {'dataset': ds, 'ablation': 'score_weight', 'param': label,
                   'k%': kp, 'k_frac': k, **m, **res}
            rows_sw.append(row)
            print(f"    cov={m['coverage']:.3f}  pur={m['purity_induced']:.4f}  "
                  f"OCR100={m['ocr_b100']:.3f}  #cases={m['n_cases']:,}  {res['time_s']:.1f}s")

df_abl_sw = pd.DataFrame(rows_sw)

# ─── Pivot table ───
print('\n=== Score-Weighted vs Uniform — pivot ===')
for ds in ['AML100k', 'AML1M']:
    print(f'\n--- {ds} ---')
    sub = df_abl_sw[df_abl_sw['dataset'] == ds]
    pivot = sub.pivot_table(
        index='k%', columns='param',
        values=['coverage', 'purity_induced', 'ocr_b100', 'n_cases'],
        aggfunc='first').round(4)
    display(pivot)

# ─── Delta coverage ───
print('\n=== Δ Coverage (score_weighted − uniform) ===')
for ds in ['AML100k', 'AML1M']:
    sub = df_abl_sw[df_abl_sw['dataset'] == ds]
    for kp in ['1%', '5%']:
        row_w = sub[(sub['param'] == 'score_weighted') & (sub['k%'] == kp)]
        row_u = sub[(sub['param'] == 'uniform')        & (sub['k%'] == kp)]
        if not row_w.empty and not row_u.empty:
            dc = float(row_w['coverage'])       - float(row_u['coverage'])
            dp = float(row_w['purity_induced']) - float(row_u['purity_induced'])
            print(f'  {ds} k={kp}: Δcov={dc:+.4f}  Δpur={dp:+.5f}')

print(f'\nAblacao Score-Weight: {len(rows_sw)} runs concluidos.')

In [ ]:
# CELULA 9 - Export
df_all = pd.concat([df_abl_dl, df_abl_b, df_abl_arch], ignore_index=True)
df_all.to_csv(ABL_OUT / 'ablation_results.csv', index=False)
print(f'CSV: {ABL_OUT}/ablation_results.csv ({len(df_all)} rows)')

# Tabela LaTeX: ablacao de arquitetura
arch_cols = ['dataset','param','k%','n_cases','coverage','purity_induced',
             'ocr_b100','time_s']
df_abl_arch[arch_cols].round(4).to_latex(
    ABL_OUT / 'ablation_architecture.tex', index=False, escape=False)

# Tabela LaTeX: ablacao de delta_L
dl_cols = ['dataset','param','k%','n_cases','coverage','purity_induced',
           'ocr_b100','time_s']
df_abl_dl[dl_cols].round(4).to_latex(
    ABL_OUT / 'ablation_delta_l.tex', index=False, escape=False)

# Tabela LaTeX: ablacao de budget
b_cols = ['dataset','budget_B','k%','n_cases','coverage','purity_induced',
          'ocr_b100','edges_per_case_max','time_s']
df_abl_b[b_cols].round(4).to_latex(
    ABL_OUT / 'ablation_budget.tex', index=False, escape=False)

print('LaTeX salvos.')
print('\nProgresso:')
print('1.[ok] nb00  2.[ok] nb01  3.[v3] nb02-BTCS  4.[ok] nb03-ablacoes')
print('5.[ ] nb04-multidataset')